# smoketest in Google Colab

This notebook builds [smoketest](https://github.com/h1ddenpr0cess20/smoketest) inside a Colab VM, launches it as a production server, and embeds it right in this notebook — the same general pattern used by Colab notebooks that spin up a UI (e.g. Unsloth Studio). It also installs LM Studio's headless server in the same VM and downloads **google/gemma-4-26b-a4b-qat** for it, plus an optional Ollama install, so smoketest's local providers have something to talk to.

**Use a GPU runtime with High-RAM** (`Runtime > Change runtime type`). smoketest itself doesn't need a GPU, but the Gemma 4 model this notebook downloads is large and the free default runtime's ~12 GB RAM is unlikely to be enough to load it.

Run the cells top to bottom. Step 7 embeds smoketest directly below its cell, authenticated as you via Colab — no public link involved. Once it's open, pick **LM Studio** or **Ollama** (or OpenAI/xAI, if you have a key) as the provider.

**Please read the caveats near the end before you start** — they cover what does and doesn't work through this setup, and the trust boundary of running a coding assistant on a shared cloud VM.

## 1. Install Node.js 22+

smoketest requires Node 22 or newer. Colab's base image doesn't ship it, so this installs it from NodeSource.

In [ ]:
%%bash
set -euo pipefail
need_install=1
if command -v node >/dev/null 2>&1; then
  major=$(node -v | sed 's/^v//' | cut -d. -f1)
  if [ "$major" -ge 22 ]; then
    need_install=0
  fi
fi
if [ "$need_install" -eq 1 ]; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
  apt-get install -y nodejs
fi
node -v
npm -v

## 2. Install LM Studio and download a model

This installs `llmster` — LM Studio's headless daemon for Linux servers — and its `lms` CLI, with no GUI involved. Because it runs in this same Colab VM as smoketest (rather than on your own computer), `127.0.0.1:1234` correctly resolves between the two, so the LM Studio provider works from the tunnel URL later on. (Ollama gets the same treatment as an optional step further down.)

In [ ]:
%%bash
set -euo pipefail
curl -fsSL https://lmstudio.ai/install.sh | bash

In [ ]:
import os
import shutil
import subprocess

lms_path = shutil.which("lms")
if lms_path is None:
    # The installer's PATH change only takes effect in new shells, not this
    # already-running kernel, so fall back to locating the binary directly.
    found = subprocess.run(
        ["bash", "-c", "find \"$HOME\" -maxdepth 6 -type f -name lms 2>/dev/null | head -n1"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    if found:
        bin_dir = os.path.dirname(found)
        os.environ["PATH"] = bin_dir + os.pathsep + os.environ["PATH"]
        lms_path = found

assert lms_path, "Could not locate the `lms` CLI after installation — check the install log above."
print(f"lms found at: {lms_path}")
!lms --version
!lms daemon up

In [ ]:
LMSTUDIO_MODEL = "google/gemma-4-26b-a4b-qat"  # @param {type:"string"}

!lms get {LMSTUDIO_MODEL} --yes
!lms ls
!df -h /content

The `lms ls` output above shows the exact model identifier LM Studio loaded — you may need to paste that (rather than `LMSTUDIO_MODEL`) into smoketest's Provider settings if they differ.

In [ ]:
import os
import subprocess
import time
import urllib.request

help_text = subprocess.run(
    ["lms", "server", "start", "--help"], capture_output=True, text=True
).stdout

# Flag support varies between lms CLI versions (some reject --host outright),
# so only pass flags this installed version actually understands. The default
# bind address (127.0.0.1) is what we want anyway: smoketest reaches LM
# Studio over 127.0.0.1 from this same VM, so it never needs to be reachable
# from outside.
cmd = ["lms", "server", "start", "--port", "1234"]
for flag in ("--cors", "--no-gui"):
    if flag in help_text:
        cmd.append(flag)
print("Running:", " ".join(cmd))

LMSTUDIO_LOG = "/content/lmstudio_server.log"
lmstudio_log_file = open(LMSTUDIO_LOG, "w")
lmstudio_server_proc = subprocess.Popen(
    cmd,
    env=os.environ.copy(),
    stdout=lmstudio_log_file,
    stderr=subprocess.STDOUT,
)
print(f"LM Studio server starting (pid {lmstudio_server_proc.pid}); logging to {LMSTUDIO_LOG}")

healthy = False
for _ in range(90):
    try:
        with urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=2) as resp:
            if resp.status == 200:
                healthy = True
                break
    except Exception:
        pass
    time.sleep(2)

if healthy:
    print("LM Studio server is up on 127.0.0.1:1234.")
else:
    print(f"LM Studio server did not report healthy in time — check {LMSTUDIO_LOG}:")
    !tail -n 80 {LMSTUDIO_LOG}

## 3. (Optional) Install Ollama

Ollama is smoketest's other local-provider option. Installing it here, in this same Colab VM, means `127.0.0.1:11434` correctly resolves between it and smoketest — just like LM Studio above. Skip this section if you only want LM Studio.

In [ ]:
%%bash
set -euo pipefail
# The official install.sh sets up a systemd service, which fails on Colab
# (no systemd/PID 1 running), so install the binary directly instead — we
# start and manage `ollama serve` ourselves in the next cell. Downloading
# first (rather than curl | tar) means a failed download shows curl's own
# error instead of a confusing downstream tar crash.
curl --retry 3 --retry-delay 2 -4 -fSL -o /tmp/ollama-linux-amd64.tgz \
  https://ollama.com/download/ollama-linux-amd64.tgz
tar zxf /tmp/ollama-linux-amd64.tgz -C /usr
rm -f /tmp/ollama-linux-amd64.tgz
ollama --version

In [ ]:
import os
import subprocess
import time
import urllib.request

OLLAMA_LOG = "/content/ollama_server.log"
ollama_log_file = open(OLLAMA_LOG, "w")
ollama_server_proc = subprocess.Popen(
    ["ollama", "serve"],
    env=os.environ.copy(),
    stdout=ollama_log_file,
    stderr=subprocess.STDOUT,
)
print(f"Ollama server starting (pid {ollama_server_proc.pid}); logging to {OLLAMA_LOG}")

healthy = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as resp:
            if resp.status == 200:
                healthy = True
                break
    except Exception:
        pass
    time.sleep(1)

if healthy:
    print("Ollama server is up on 127.0.0.1:11434.")
else:
    print(f"Ollama server did not report healthy in time — check {OLLAMA_LOG}:")
    !tail -n 50 {OLLAMA_LOG}

In [ ]:
OLLAMA_MODEL = ""  # @param {type:"string"}

if OLLAMA_MODEL:
    !ollama pull {OLLAMA_MODEL}
    print(f"Pulled {OLLAMA_MODEL}. Enter this exact name as the model in smoketest's Ollama provider settings.")
else:
    print(
        "No OLLAMA_MODEL set — skipping the pull. Set it above and rerun this cell, "
        "or run `ollama pull <model>` yourself, then enter that model name in smoketest's Provider settings."
    )

## 4. Clone smoketest

Re-running this cell on an already-cloned checkout pulls the latest commit on the chosen branch instead of cloning again.

In [ ]:
import os

REPO_URL = "https://github.com/h1ddenpr0cess20/smoketest.git"  # @param {type:"string"}
BRANCH = "master"  # @param {type:"string"}
REPO_DIR = "/content/smoketest"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch --depth 1 origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}

## 5. Install dependencies and build a production bundle

This mirrors the project's own `Dockerfile`: install with `npm ci`, then `npm run build`. It takes roughly a minute.

In [ ]:
%%bash
set -euo pipefail
if ! command -v cloudflared >/dev/null 2>&1; then
  curl -fsSL -o /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x /usr/local/bin/cloudflared
fi
cloudflared --version

## 6. Launch the smoketest server

Runs the standalone production server (like the Dockerfile's `CMD`) on `0.0.0.0:3000` in the background and waits for `/api/health` to respond.

In [ ]:
import os
import subprocess
import time
import urllib.request

APP_DIR = "/content/smoketest"
STANDALONE_DIR = os.path.join(APP_DIR, ".next/standalone")

!rm -rf {STANDALONE_DIR}/.next/static {STANDALONE_DIR}/public
!cp -r {APP_DIR}/.next/static {STANDALONE_DIR}/.next/static
!cp -r {APP_DIR}/public {STANDALONE_DIR}/public

server_env = os.environ.copy()
server_env.update(
    {
        "HOSTNAME": "0.0.0.0",
        "PORT": "3000",
        "NODE_ENV": "production",
        "NEXT_TELEMETRY_DISABLED": "1",
    }
)

SERVER_LOG = "/content/smoketest_server.log"
server_log_file = open(SERVER_LOG, "w")
server_proc = subprocess.Popen(
    ["node", "server.js"],
    cwd=STANDALONE_DIR,
    env=server_env,
    stdout=server_log_file,
    stderr=subprocess.STDOUT,
)
print(f"smoketest server starting (pid {server_proc.pid}); logging to {SERVER_LOG}")

healthy = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:3000/api/health", timeout=2) as resp:
            if resp.status == 200:
                healthy = True
                break
    except Exception:
        pass
    time.sleep(1)

if healthy:
    print("smoketest is up on 127.0.0.1:3000.")
else:
    print(f"smoketest did not report healthy in time — check {SERVER_LOG}:")
    !tail -n 50 {SERVER_LOG}

## 7. View smoketest right here in the notebook

This is the trick behind Colab notebooks for tools like Unsloth Studio: `google.colab.output.serve_kernel_port_as_iframe` proxies port 3000 through Colab's own authenticated infrastructure and embeds it below — gated by your Google login, not a public link. Re-run this cell any time you want to reopen or refresh the embedded window.

In [ ]:
from google.colab import output as colab_output

colab_output.serve_kernel_port_as_iframe(3000, path="/", height=800)

## 8. (Optional) Also expose it with a public URL

The iframe above only works inside this notebook tab, authenticated as you. If you also want a link you can open on another device or browser (e.g. your phone), this opens a [Cloudflare quick tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) to smoketest's port 3000. No account or API key is needed. The URL is random and temporary, and dies with this runtime. (LM Studio's port 1234 and Ollama's port 11434 are not tunneled — smoketest's server reaches them over `127.0.0.1` internally, so they never need to be public.) Skip this section if the embedded view above is enough.

In [ ]:
%%bash
set -e
if ! command -v cloudflared >/dev/null 2>&1; then
  curl -fsSL -o /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x /usr/local/bin/cloudflared
fi
cloudflared --version

In [ ]:
import re
import subprocess
import time

TUNNEL_LOG = "/content/cloudflared.log"
tunnel_log_file = open(TUNNEL_LOG, "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
    stdout=tunnel_log_file,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(60):
    time.sleep(1)
    with open(TUNNEL_LOG) as f:
        contents = f.read()
    match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", contents)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f"smoketest is live at: {public_url}")
    print("Open it, choose LM Studio (or OpenAI/xAI, with your own key) under Provider settings.")
else:
    print(f"Tunnel URL not found yet — check {TUNNEL_LOG}:")
    !tail -n 50 {TUNNEL_LOG}

## Caveats

- **LM Studio and Ollama both work here.** This notebook installs LM Studio's headless server, and optionally Ollama, in the same Colab VM as smoketest, so `127.0.0.1:1234` and `127.0.0.1:11434` correctly resolve between them — pick **LM Studio** or **Ollama** as the provider. OpenAI and xAI work too, with your own API key.
- **The embedded view (step 7) is the safer default.** It's proxied through Colab's own authenticated infrastructure, so only you (logged into this Colab session) can reach it. Step 8's Cloudflare tunnel is public and optional — only run it if you specifically need a link for another device.
- **The Gemma 4 model is large and memory-hungry.** The download is many gigabytes, and loading it needs real RAM/VRAM. Use a GPU + High-RAM Colab runtime — the free default runtime's ~12 GB RAM is unlikely to be enough, and even if it fits, CPU-only inference on a model this size will be slow. Check the size `lms get` reports and `df -h` output in step 2 before assuming it'll fit. The same applies if you pull a large Ollama model in the optional step.
- **LM Studio's headless Linux mode is newer and less battle-tested than the desktop app.** If any `lms` command fails, check `/content/lmstudio_server.log` and LM Studio's own headless-mode docs; behavior here is more likely to shift between LM Studio releases than the rest of this notebook.
- **API keys transit the VM.** Keys you type into Provider settings for OpenAI/xAI live in your browser's `localStorage` and are never written to disk on the VM, but each request does pass through this Colab VM on its way to the provider (and through the Cloudflare tunnel too, if you're using step 8). Don't use this setup with a key you wouldn't want touching shared cloud infrastructure. (This doesn't apply to LM Studio or Ollama — no key leaves the VM.)
- **If you do use the step 8 tunnel link, it's public but unguessable.** Anyone who has the `trycloudflare.com` URL can use your running instance (and any API key you've entered) until the tunnel closes. Don't share it.
- **Everything is ephemeral.** Closing the tab or losing the Colab runtime kills smoketest, LM Studio, Ollama, and any tunnel, and the transcript/settings living in that browser tab's `localStorage` go with it. Rerun from step 4 onward (or step 6 onward, if the checkout and build are still present) to relaunch smoketest; downloaded/pulled models persist on the VM's disk for the life of the runtime, so re-running `lms get` or `ollama pull` will skip re-downloading them.

## Stop everything

Run this when you're done, or before relaunching.

In [ ]:
import subprocess

for proc_name in ("tunnel_proc", "server_proc", "lmstudio_server_proc", "ollama_server_proc"):
    proc = globals().get(proc_name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        print(f"stopped {proc_name} (pid {proc.pid})")

subprocess.run(["lms", "server", "stop"], capture_output=True)
subprocess.run(["lms", "daemon", "down"], capture_output=True)
print("done.")